# 🔍 Punto 3 — Interpretabilidad con Grad-CAM
## Examen Parcial: Redes Neuronales y Aprendizaje Profundo

Este notebook implementa **Gradient-weighted Class Activation Mapping (Grad-CAM)** sobre ambos modelos entrenados en el Punto 1, mostrando qué regiones de textura activan la clasificación de daño.

| Ítem | Detalle |
|------|---------|
| **Modelos analizados** | TireCNN (desde cero) y EfficientNet-B0 (Transfer Learning) |
| **Técnica** | Grad-CAM (Selvaraju et al., 2017) |
| **Capas objetivo** | Última capa convolucional de cada arquitectura |
| **Análisis** | Mapas de activación en ambas clases + discusión de patrones |

> **Prerrequisito:** Haber ejecutado `tire_classification.ipynb` y tener guardados  
> `/content/TireCNN_final.pth` y `/content/EfficientNet_final.pth`

---
## 0. Instalación y configuración

In [ ]:
!pip install -q kagglehub torchvision matplotlib seaborn scikit-learn grad-cam

In [ ]:
import os, random, shutil
from pathlib import Path
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split

# pytorch-grad-cam
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ── Reproducibilidad ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE   = 224
DATA_DIR   = Path('/content/dataset_bruto/Tire Textures')
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f'Dispositivo: {DEVICE}')

---
## 1. Reconstrucción de los modelos y carga de pesos

In [ ]:
# ── Arquitectura TireCNN (idéntica al Punto 1) ────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if pool: layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class TireCNN(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32), ConvBlock(32,  64),
            ConvBlock(64, 128), ConvBlock(128, 256),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.global_avg_pool(x)
        return self.classifier(x)


# ── Cargar dataset para obtener CLASS_NAMES y split ───────────────────────────
raw_dataset = datasets.ImageFolder(root=str(DATA_DIR), transform=None)
CLASS_NAMES = raw_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
targets     = [s[1] for s in raw_dataset.samples]
indices     = list(range(len(raw_dataset)))

train_idx, temp_idx = train_test_split(indices, test_size=0.30,
                                        stratify=targets, random_state=SEED)
temp_tgts = [targets[i] for i in temp_idx]
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50,
                                      stratify=temp_tgts, random_state=SEED)
print(f'Clases: {CLASS_NAMES}  |  Test: {len(test_idx)} imágenes')


# ── Cargar pesos guardados ────────────────────────────────────────────────────
def load_tirecnn(path, num_classes, device):
    model = TireCNN(num_classes=num_classes)
    ckpt  = torch.load(path, map_location=device)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()
    return model.to(device)

def load_efficientnet(path, num_classes, device):
    model = models.efficientnet_b0(weights=None)
    in_f  = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_f, 256), nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),    nn.Linear(256, num_classes),
    )
    ckpt  = torch.load(path, map_location=device)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()
    return model.to(device)

model_scratch = load_tirecnn('/content/TireCNN_final.pth',      NUM_CLASSES, DEVICE)
model_effnet  = load_efficientnet('/content/EfficientNet_final.pth', NUM_CLASSES, DEVICE)
print('✅ Ambos modelos cargados correctamente.')

---
## 2. Identificación de capas objetivo para Grad-CAM

Grad-CAM requiere la **última capa convolucional** del backbone, ya que es donde las activaciones
tienen la mayor resolución semántica antes de colapsar al vector de features.

In [ ]:
# ── TireCNN: última conv está en features[3].block[3] ─────────────────────────
# (ConvBlock 4: segundo Conv2d antes del MaxPool)
target_layers_scratch = [model_scratch.features[-1].block[-2]]  # BN de la última conv
# Más robusto: apuntar al último Conv2d directamente
last_conv_scratch = None
for layer in model_scratch.features[-1].block:
    if isinstance(layer, nn.Conv2d):
        last_conv_scratch = layer
target_layers_scratch = [last_conv_scratch]
print(f'TireCNN   — capa objetivo: {last_conv_scratch}')

# ── EfficientNet-B0: última capa convolucional del backbone ───────────────────
# En EfficientNet-B0, el último bloque MBConv está en model.features[-1]
# La conv final es model.features[-1][0] (pointwise conv del último bloque)
last_conv_effnet = model_effnet.features[-1][0]  # Conv2d 1x1 de expansión final
target_layers_effnet = [last_conv_effnet]
print(f'EfficientNet — capa objetivo: {last_conv_effnet}')

---
## 3. Utilidades de Grad-CAM

In [ ]:
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def pil_to_tensor(img_pil: Image.Image) -> torch.Tensor:
    """PIL → tensor normalizado (1, 3, H, W)."""
    return val_tf(img_pil).unsqueeze(0)

def tensor_to_rgb(tensor: torch.Tensor) -> np.ndarray:
    """Tensor normalizado → numpy RGB [0,1] para overlay."""
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    img  = (tensor.squeeze(0).cpu() * std + mean).clamp(0, 1)
    return img.permute(1, 2, 0).numpy().astype(np.float32)


def compute_gradcam(model, target_layers, img_tensor, target_class=None):
    """
    Calcula el mapa Grad-CAM para una imagen.

    Args:
        model         : modelo PyTorch en eval mode
        target_layers : lista con la capa objetivo
        img_tensor    : (1, 3, H, W) tensor normalizado
        target_class  : índice de clase objetivo. None → clase predicha.

    Returns:
        cam_image     : np.ndarray (H, W, 3) RGB con overlay coloreado
        grayscale_cam : np.ndarray (H, W) mapa de calor en escala de grises [0,1]
        pred_class    : int con la clase predicha
        pred_conf     : float con la confianza de la predicción
    """
    with GradCAM(model=model, target_layers=target_layers) as cam:
        targets = [ClassifierOutputTarget(target_class)] if target_class is not None else None
        grayscale_cam = cam(input_tensor=img_tensor.to(DEVICE), targets=targets)
    grayscale_cam = grayscale_cam[0]   # (H, W)

    # Predicción
    with torch.no_grad():
        logits     = model(img_tensor.to(DEVICE))
        probs      = torch.softmax(logits, dim=1).cpu().squeeze()
        pred_class = probs.argmax().item()
        pred_conf  = probs[pred_class].item()

    rgb_img   = tensor_to_rgb(img_tensor)  # (H, W, 3) float32 [0,1]
    cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
    return cam_image, grayscale_cam, pred_class, pred_conf, probs.numpy()


print('✅ Utilidades Grad-CAM listas.')

---
## 4. Selección de imágenes de muestra — ambas clases

In [ ]:
# Seleccionar N imágenes correctamente clasificadas por clase del conjunto de test
N_SAMPLES = 5   # imágenes por clase

samples_by_class = {cls: [] for cls in range(NUM_CLASSES)}

for idx in test_idx:
    img_path, true_label = raw_dataset.samples[idx]
    img_pil    = Image.open(img_path).convert('RGB')
    img_tensor = pil_to_tensor(img_pil)

    with torch.no_grad():
        logits     = model_effnet(img_tensor.to(DEVICE))
        pred_label = logits.argmax(1).item()

    # Solo imágenes correctamente clasificadas para análisis limpio
    if pred_label == true_label and len(samples_by_class[true_label]) < N_SAMPLES:
        samples_by_class[true_label].append({
            'path': img_path,
            'true': true_label,
            'img_pil': img_pil,
            'img_tensor': img_tensor,
        })

    if all(len(v) >= N_SAMPLES for v in samples_by_class.values()):
        break

for cls_idx, samples in samples_by_class.items():
    print(f'Clase {CLASS_NAMES[cls_idx]:>12}: {len(samples)} imágenes seleccionadas')

---
## 5. Visualización Grad-CAM — por clase y por modelo

In [ ]:
def plot_gradcam_grid(model, target_layers, model_name,
                      samples_by_class, class_names, save_path):
    """
    Genera una grilla con 3 columnas por imagen:
      [Imagen original] [Mapa de calor] [Overlay Grad-CAM]
    Una fila por cada muestra, agrupadas por clase.
    """
    all_samples = []
    for cls_idx in sorted(samples_by_class.keys()):
        for s in samples_by_class[cls_idx]:
            all_samples.append(s)

    n_rows = len(all_samples)
    fig, axes = plt.subplots(n_rows, 3, figsize=(13, 4 * n_rows))
    if n_rows == 1: axes = axes[np.newaxis, :]

    col_titles = ['Imagen original', 'Mapa Grad-CAM (escala de grises)', 'Overlay Grad-CAM']
    for col, ct in enumerate(col_titles):
        axes[0][col].set_title(ct, fontsize=11, fontweight='bold', pad=8)

    prev_class = None
    for row, sample in enumerate(all_samples):
        cls_idx   = sample['true']
        cls_name  = class_names[cls_idx]
        img_pil   = sample['img_pil']
        img_tensor= sample['img_tensor']

        # Línea divisora entre clases
        if prev_class is not None and cls_idx != prev_class:
            for col in range(3):
                axes[row][col].axhline(y=-5, color='black', lw=3)
        prev_class = cls_idx

        cam_img, gray_cam, pred, conf, probs = compute_gradcam(
            model, target_layers, img_tensor, target_class=cls_idx
        )

        # ── Columna 0: imagen original ──────────────────────────────────────
        img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
        axes[row][0].imshow(img_resized)
        axes[row][0].set_ylabel(
            f'Clase real:\n{cls_name}',
            fontsize=9, fontweight='bold', rotation=0,
            labelpad=80, va='center'
        )
        pred_name = class_names[pred]
        color = '#27ae60' if pred == cls_idx else '#e74c3c'
        axes[row][0].set_xlabel(
            f'Predicho: {pred_name} ({conf*100:.1f}%)',
            fontsize=8, color=color, fontweight='bold'
        )
        axes[row][0].axis('off')

        # ── Columna 1: mapa de calor en escala de grises ────────────────────
        im = axes[row][1].imshow(gray_cam, cmap='jet', vmin=0, vmax=1)
        plt.colorbar(im, ax=axes[row][1], fraction=0.046, pad=0.04)
        axes[row][1].axis('off')

        # ── Columna 2: overlay ──────────────────────────────────────────────
        axes[row][2].imshow(cam_img)
        axes[row][2].axis('off')

        # Barra de probabilidades en el overlay
        prob_text = '  '.join(
            [f'{class_names[i]}: {p*100:.1f}%' for i, p in enumerate(probs)]
        )
        axes[row][2].set_xlabel(prob_text, fontsize=7.5, color='#2c3e50')

    plt.suptitle(
        f'Grad-CAM — {model_name}\nActivaciones por clase (target = clase real)',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Guardado: {save_path}')


# ── TireCNN ───────────────────────────────────────────────────────────────────
plot_gradcam_grid(
    model_scratch, target_layers_scratch, 'TireCNN (desde cero)',
    samples_by_class, CLASS_NAMES,
    '/content/gradcam_TireCNN.png'
)

# ── EfficientNet-B0 ───────────────────────────────────────────────────────────
plot_gradcam_grid(
    model_effnet, target_layers_effnet, 'EfficientNet-B0 (Transfer Learning)',
    samples_by_class, CLASS_NAMES,
    '/content/gradcam_EfficientNet.png'
)

---
## 6. Grad-CAM contrafactual — mismo modelo, distintas clases objetivo

Forzamos Grad-CAM a responder: *¿qué regiones activaría el modelo si creyera que esta imagen es de la clase contraria?*  
Esto revela si el modelo tiene representaciones internas diferenciadas para cada clase.

In [ ]:
def plot_contrastive_gradcam(model, target_layers, model_name,
                              sample, class_names, save_path):
    """
    Para una misma imagen muestra los mapas Grad-CAM activados hacia
    cada clase (correcta y contrafactual), evidenciando la discriminación
    interna del modelo.
    """
    img_pil    = sample['img_pil']
    img_tensor = sample['img_tensor']
    true_cls   = sample['true']
    n_classes  = len(class_names)

    fig, axes = plt.subplots(1, n_classes + 1, figsize=(5 * (n_classes + 1), 5))

    # Imagen original
    axes[0].imshow(img_pil.resize((IMG_SIZE, IMG_SIZE)))
    axes[0].set_title(f'Original\n(clase real: {class_names[true_cls]})',
                      fontweight='bold')
    axes[0].axis('off')

    for cls_idx in range(n_classes):
        cam_img, _, pred, conf, _ = compute_gradcam(
            model, target_layers, img_tensor, target_class=cls_idx
        )
        tag = '✅ (correcta)' if cls_idx == true_cls else '⚠️ (contrafactual)'
        axes[cls_idx + 1].imshow(cam_img)
        axes[cls_idx + 1].set_title(
            f'Grad-CAM → clase: {class_names[cls_idx]}\n{tag}',
            fontsize=9, fontweight='bold'
        )
        axes[cls_idx + 1].axis('off')

    plt.suptitle(f'Análisis contrafactual — {model_name}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Usar una muestra de cada clase
for cls_idx, samples in samples_by_class.items():
    if samples:
        plot_contrastive_gradcam(
            model_effnet, target_layers_effnet, 'EfficientNet-B0',
            samples[0], CLASS_NAMES,
            f'/content/gradcam_contrastive_clase{cls_idx}.png'
        )

---
## 7. Comparación Grad-CAM vs Grad-CAM++ vs EigenCAM

Evaluamos tres variantes de CAM para validar la robustez de las activaciones encontradas.

In [ ]:
def compare_cam_methods(model, target_layers, sample, class_names, save_path):
    """
    Muestra GradCAM, GradCAM++ y EigenCAM sobre la misma imagen
    para verificar consistencia de las activaciones.
    """
    img_pil    = sample['img_pil']
    img_tensor = sample['img_tensor']
    true_cls   = sample['true']
    rgb_img    = tensor_to_rgb(img_tensor)

    cam_methods = {
        'GradCAM':    GradCAM,
        'GradCAM++':  GradCAMPlusPlus,
        'EigenCAM':   EigenCAM,
    }

    fig, axes = plt.subplots(1, len(cam_methods) + 1,
                              figsize=(5 * (len(cam_methods) + 1), 5))
    axes[0].imshow(img_pil.resize((IMG_SIZE, IMG_SIZE)))
    axes[0].set_title(f'Original\nClase: {class_names[true_cls]}',
                      fontweight='bold')
    axes[0].axis('off')

    target = [ClassifierOutputTarget(true_cls)]
    for col, (name, CamClass) in enumerate(cam_methods.items(), start=1):
        with CamClass(model=model, target_layers=target_layers) as cam:
            gray = cam(input_tensor=img_tensor.to(DEVICE), targets=target)[0]
        overlay = show_cam_on_image(rgb_img, gray, use_rgb=True)
        axes[col].imshow(overlay)
        axes[col].set_title(name, fontweight='bold')
        axes[col].axis('off')

    plt.suptitle('Comparación de métodos CAM — EfficientNet-B0',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Aplicar sobre la primera muestra disponible
first_sample = list(samples_by_class.values())[0][0]
compare_cam_methods(
    model_effnet, target_layers_effnet,
    first_sample, CLASS_NAMES,
    '/content/gradcam_methods_comparison.png'
)

---
## 8. Análisis cuantitativo — cobertura del mapa de activación

Medimos qué fracción del área de la imagen concentra la activación Grad-CAM
y si difiere entre clases (una llanta dañada debería concentrar la activación
en zonas más localizadas que una llanta en buen estado).

In [ ]:
def activation_stats(model, target_layers, samples_by_class, class_names,
                     threshold=0.5):
    """
    Para cada clase calcula:
    - Cobertura: % del área de la imagen con activación > threshold
    - Intensidad media del mapa de calor
    - Entropía del mapa (dispersión de la activación)
    """
    stats = {cls: {'coverage': [], 'mean_intensity': [], 'entropy': []}
             for cls in range(len(class_names))}

    for cls_idx, samples in samples_by_class.items():
        for s in samples:
            _, gray, _, _, _ = compute_gradcam(
                model, target_layers, s['img_tensor'], target_class=cls_idx
            )
            total_pixels = gray.size
            coverage     = (gray > threshold).sum() / total_pixels
            mean_int     = gray.mean()
            # Entropía de Shannon normalizada
            hist, _      = np.histogram(gray, bins=20, range=(0,1), density=True)
            hist         = hist + 1e-10
            entropy      = -np.sum(hist * np.log2(hist)) / np.log2(len(hist))

            stats[cls_idx]['coverage'].append(coverage)
            stats[cls_idx]['mean_intensity'].append(mean_int)
            stats[cls_idx]['entropy'].append(entropy)

    return stats


print('Calculando estadísticas de activación...')
stats_effnet  = activation_stats(model_effnet,  target_layers_effnet,
                                  samples_by_class, CLASS_NAMES)
stats_scratch = activation_stats(model_scratch, target_layers_scratch,
                                  samples_by_class, CLASS_NAMES)

# Tabla resumen
import pandas as pd
rows = []
for model_name, stats in [('EfficientNet-B0', stats_effnet),
                            ('TireCNN', stats_scratch)]:
    for cls_idx, cls_stats in stats.items():
        rows.append({
            'Modelo':          model_name,
            'Clase':           CLASS_NAMES[cls_idx],
            'Cobertura media': f"{np.mean(cls_stats['coverage'])*100:.1f}%",
            'Intensidad media':f"{np.mean(cls_stats['mean_intensity']):.4f}",
            'Entropía media':  f"{np.mean(cls_stats['entropy']):.4f}",
        })

df_stats = pd.DataFrame(rows)
print('\n📊 Estadísticas de cobertura Grad-CAM:')
print(df_stats.to_string(index=False))
df_stats.to_csv('/content/gradcam_stats.csv', index=False)

In [ ]:
# Visualizar estadísticas
metrics_plot = ['coverage', 'mean_intensity', 'entropy']
labels_plot  = ['Cobertura (fracción)', 'Intensidad media', 'Entropía normalizada']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

x = np.arange(len(CLASS_NAMES))
width = 0.3
colors_models = {'EfficientNet-B0': '#2980b9', 'TireCNN': '#e74c3c'}

for col, (metric, label) in enumerate(zip(metrics_plot, labels_plot)):
    for i, (model_name, stats) in enumerate(
        [('EfficientNet-B0', stats_effnet), ('TireCNN', stats_scratch)]
    ):
        vals = [np.mean(stats[cls][metric]) for cls in range(len(CLASS_NAMES))]
        errs = [np.std(stats[cls][metric])  for cls in range(len(CLASS_NAMES))]
        axes[col].bar(x + i * width, vals, width, yerr=errs,
                       label=model_name, color=colors_models[model_name],
                       alpha=0.8, capsize=4)
    axes[col].set_title(label, fontweight='bold')
    axes[col].set_xticks(x + width / 2)
    axes[col].set_xticklabels(CLASS_NAMES, rotation=15, ha='right')
    axes[col].legend(fontsize=8)
    axes[col].grid(axis='y', alpha=0.3)

plt.suptitle('Estadísticas cuantitativas de activación Grad-CAM por clase y modelo',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/gradcam_stats_plot.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Guardar y descargar

In [ ]:
from google.colab import files

outputs = [
    '/content/gradcam_TireCNN.png',
    '/content/gradcam_EfficientNet.png',
    '/content/gradcam_contrastive_clase0.png',
    '/content/gradcam_contrastive_clase1.png',
    '/content/gradcam_methods_comparison.png',
    '/content/gradcam_stats_plot.png',
    '/content/gradcam_stats.csv',
]
for f_path in outputs:
    if os.path.exists(f_path):
        files.download(f_path)
    else:
        print(f'⚠️  No encontrado: {f_path}')
print('✅ Descargas iniciadas.')

---
## 📝 Guía de interpretación para el informe

### Qué es Grad-CAM
Grad-CAM (Selvaraju et al., ICCV 2017) pondera cada mapa de activación de la última capa convolucional por el gradiente de la clase objetivo, produciendo un mapa de saliencia que indica qué regiones espaciales fueron más relevantes para la decisión.

### Qué buscar en los mapas

**Llantas dañadas (clase positiva):**
- El modelo debería activar en zonas de **grietas, desgaste irregular, surcos rotos o bordes deformados**
- Alta concentración → el modelo aprendió características locales específicas del daño
- Activación dispersa → el modelo usa contexto global (menos deseable)

**Llantas en buen estado (clase negativa):**
- Activación más **uniforme y distribuida** sobre la textura
- Patrón regular de surcos → el modelo reconoce la uniformidad como señal de buen estado

### Comparación entre modelos
| Aspecto | TireCNN | EfficientNet-B0 |
|---------|---------|------------------|
| Precisión de localización | Menor (menos filtros) | Mayor (MBConv + SE blocks) |
| Cobertura esperada | Mayor (más difusa) | Menor (más localizada) |
| Robustez del mapa | Puede ser ruidoso | Más suave y consistente |

### Variantes CAM evaluadas
- **GradCAM**: pondera por gradiente global promediado → rápido, puede ser ruidoso
- **GradCAM++**: pondera por derivadas de orden superior → mejor localización de objetos múltiples
- **EigenCAM**: usa el primer vector propio de las activaciones → no requiere gradientes (más estable)

### Métrica cuantitativa
- **Cobertura alta** en clase dañada → el modelo usa área extensa → posible señal de confusión
- **Cobertura baja** en clase dañada → el modelo aprendió a localizar el daño con precisión
- **Entropía alta** → activación dispersa; **Entropía baja** → activación concentrada